## 기본구조

In [ ]:
%pip install langchain langchain-google-genai langchain-anthropic langchain-community langchain-core langgraph
%pip install python-dotenv chromadb faiss-cpu google-genai

## prompt template

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

In [ ]:
load_dotenv()

api_key_value = os.getenv("GEMINI_API_KEY")

if not api_key_value:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

# LangChain이 GOOGLE_API_KEY를 알아볼 수 있도록 맞춰둔다.
# LangChain의 Gemini 연동 라이브러리는 보통 내부적으로 환경변수 이름 GOOGLE_API_KEY를 찾아서 사용하기에 미리 맞춘다.
os.environ["GOOGLE_API_KEY"] = api_key_value

# 예를 들어   llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")  라는 코드를  실행하면 이 코드가 내부적으로 GOOGLE_API_KEY를 찾아서 인증에 사용한다. 

In [ ]:
# 우리의 api key를 활용해서 Gemini 클라이언트를 만든다.(인스턴스 생성) 

client = genai.Client(api_key=api_key)


# Gemini 모델에 실제 요청을 보낸다. model  = 사용할 Gemini 모델 이름 / content = 모델에게 보낼 입력 문장이다. 

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="안녕 이름이 뭐니?",
    )
    print("Gemini API call succeeded")
    print(response.text)   # response는 우리의 api 키로 불러온 Gemini 모델의 답변이다. 
except Exception as e:
    print("Gemini API call failed")
    print(type(e).__name__)
    print(e)

## Langchain

langchain은 위와 같이 LLM을 그냥 한 번 호출하는 수준을 넘어서, LLM 기반 애플리케이션을 만들기 쉽게 해주는 파이썬 라이브러리이다. 

  - ChatGoogleGenerativeAI는 Gemini 채팅 모델을 LangChain 인터페이스로 감싼 객체입니다.
  - SystemMessage, HumanMessage는 대화 구조를 명확하게 표현하는 메시지 타입입니다.
  - llm.invoke(messages)는 이 메시지들을 모델에 보내고 응답을 받는 표준 호출 방식입니다.

  Langchain을 사용하면  

  - 프롬프트를 구조적으로 관리
  - 여러 LLM 제공자(Gemini, Anthropic 등)를 비슷한 방식으로 사용
  - 대화 기록, 메시지, 출력 파싱 관리
  - 외부 도구 호출
    예: 검색, 계산기, DB 조회, API 호출
  - RAG 구성
    예: PDF/문서 검색 후 그 내용을 바탕으로 답변
  - 체인/에이전트 워크플로우 구성
    예: 질문 분해 → 정보 조회 → 답변 생성


## 한 줄로 정리하면, LangChain은 “LLM을 활용한 실제 서비스나 자동화 프로그램을 만들 때 구조를 잡아주는 라이브러리”

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI # LangChain에서 Google Gemini 모델을 쓰기 위한 클래스를 가져오는 코드. 즉, Gemini를 랭체인 방식으로 호출하기 위한 라이브러리이다. 
from langchain_core.messages import HumanMessage, SystemMessage # 이 두개를 구분해주면, human에 해당하는 부분이 내가 할 질문이고 system은 llm에게 주는 프롬프트이다. 

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 가장 기본 호출
messages = [
    # personalize system message (LLM한테 부여하는 메세지이다. 가이드를 준 것이다.)
    SystemMessage(content="너는 친절한 어시스턴트야. 주요 역할은 질문에 따라서 답변과 설명을 제공하는 것이야."),

    # user message(HumanMessage는 내가 한 질문이다. )
    HumanMessage(content="데이터사이언스 분야에서 가장 핫한 개념 3가지만 말해줘")
]
response = llm.invoke(messages) # invoke
print(response.content)

원래 Gemini API를 직접 쓰면:

```python
from google import genai

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
model="gemini-2.5-flash",
contents="안녕"
)
```

  이런 식으로 호출합니다.

  LangChain을 쓰면:

```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = llm.invoke("안녕")
```

  이런 식으로 사용할 수 있습니다(이 경우 api key를 직접 코드에 넣지 않고 환경변수에서 자동으로 읽는다. )

  즉, ChatGoogleGenerativeAI는 Gemini를 LangChain의 llm.invoke(), 체인(chain), 프롬프트 템플릿, RAG 같은 구조 안에서 쉽게 연결할 수 있게 해주는 역할을 합니다.

In [ ]:
from langchain_core.prompts import PromptTemplate # 말 그대로 프롬프트의 템플릿이다. 

# template 정의. {country}는 변수로, 이후에 값이 들어갈 자리를 의미 (내가 지정할 변수를 fstring으로 처리함)
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
#prompt.format(country="미국")

chain = prompt | llm #랭체인의 체인닝이다. 이렇게 하면? 프롬프트와 LLM이 체이닝이 된 것이다. 말그대로 내가 질문을 할 때 프롬프트를 거쳐서 LLM에 입력한다는 것이다. 프롬프트와 LLM을 결합한것이다. 

response = chain.invoke({"일본"})
print(response.content)


# 랭체인에서는 |을 통해 chaining을 할 수 있다

message에 지정하는 방식과 프롬프트 템플릿을 만드는 방식은 아래와 같이 동시에 사용할 수 있다.


```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
("system", "너는 친절한 어시스턴트야. 질문에 대해 쉽게 설명해줘."),
("human", "{country}의 수도는 어디인가요?")
])

chain = prompt | llm

response = chain.invoke({"country": "일본"})
print(response.content)
```


즉, ChatPromptTemplate을 통해 시스템 프롬프트(+휴먼 프롬프트) + 사용자의 질문 템플릿까지 같이 사용 가능하다. 그리고 보통 이 방식이 message 안에 지정하는 방식보다 더 자연스럽다. 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate # langchain core라는 모듈 안에 프롬프트 템플릿이라는 함수가 있다. 말그대로 프롬프트의 템플릿으로써 내가 템플릿 정의를 내가 지정할 변수를 다음처럼 템플릿으로 변수를 지정할 수 있다. 
# 챗 프롬프트 템플릿은 역할을 기반해서 상호작용을 위해 설계된 템플릿이다. 질문과 대화가 오가는! 


chat_template = ChatPromptTemplate.from_messages(   # 시스템에게 어떠한 역할을 줄 수 있다. 대화형 LLM에서 대화 구조 흐름을 정의하는 3가지 핵심 요소가 있다)
    [
        # role, message
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name} 입니다."),
        #("human", "반가워요!"),  
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"), # AI 응답 메시지
        # 사용자 입력 메시지
        ("human", "{user_input}"),
    ]
)

# 챗 message 를 생성합니다. 
# LLM에서 대화의 흐름을 정의하는 부분. system messaage 부분은 즉, AI의 페르소나를 정의하는 첫 번쨰 메세지이다. 모델이 대화내내 지켜야할 가이드 등을 포함한다. 
# ai 메세지는 이전 사람이 입력한 질문에 대한 모델의 반응, 대화 기억의 일부임(이전에 AI가 했던 응답을 메시지 프롬프트 안에 넣는 역할 즉, 프롬프트에 이런 대화가 이미 있었다고 가정한 부분임)
# human 메세지는 사용자가 LLM에게 질문한 부분으로 모델이 반응해야 할 대상이다. 

messages = chat_template.format_messages(
    name="메타코드 챗봇", user_input="당신의 이름은 무엇입니까?"
)
messages

# 이미 만들어진 경우에는 체인을 쓰지 않고 바로 llm.invoke(messages)를 한다,. 
messages = chat_template.format_messages(
    name="메타코드 챗봇",
    user_input="당신의 이름은 무엇입니까?"
)

response = llm.invoke(messages)
print(response.content)


# 이 코드는 LLM에게 보낼 메세지 목록을 미리 만든 것이다. 실제로 답변 받으려면 llm.invoke()를 해야겠지 또는 체인으로 연결해서 쓰거나!

In [ ]:
# MessagesPlaceholder를 사용하면, 시스템 메시지와 현재 사용자 입력 사이에 이전 대화 기록 전체를 유연하게 삽입할 수 있어서, 상태(State)를 가진 대화형 애플리케이션을 만들 때 필수적입니다.

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder  # 핵심!
)
from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

# 1. 이전 대화 기록 (History) 예시
# 이는 실제 애플리케이션에서는 Memory 컴포넌트 등에서 가져옵니다.
chat_history = [
    HumanMessage(content="안녕하세요! 저는 파이썬 개발자입니다."),
    AIMessage(content="반갑습니다. 파이썬 개발에 대해 어떤 것을 도와드릴까요?"),
]

# 2. ChatPromptTemplate 정의
# from_messages를 사용하여 템플릿의 구조를 정의합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        # 1) 시스템 지침 (System Instruction)
        # 챗봇의 역할을 정의합니다.
        SystemMessagePromptTemplate.from_template(
            "당신은 친절하고 전문적인 AI 어시스턴트입니다. 모든 답변은 한국어로 작성해 주세요."
        ),

        # 2) 대화 기록 Placeholder (MessagesPlaceholder:LLM 통해서 챗봇 만들 때 대화구조 정의하고 맥락을 구분하기 위해 사용되는 표식임)
        # 이 부분이 동적으로 'chat_history'가 들어갈 위치를 지정합니다.
        # LLM 에게 메세지를 명확히 구분할 수 있도록 한다. 대화 기록들이 일관적으로 쌓일 수 있게끔 구조화하는 역할을 한다.-> 맥락 유지에 도움
        # variable_name을 통해 나중에 format 함수에 어떤 이름으로 전달할지 지정합니다.
        MessagesPlaceholder(variable_name="history"),

        # 3) 사용자의 새로운 입력 (Current Human Input)
        # 현재 사용자가 모델에게 보내는 질문입니다.
        HumanMessagePromptTemplate.from_template("{input}")
    ]
)

# 3. Prompt Template에 실제 값 바인딩 및 최종 PromptValue 생성
# prompt.invoke() 또는 prompt.format_prompt()를 사용하여 값을 채웁니다.
final_prompt_value = prompt.invoke(
    {
        "history": chat_history,  # MessagesPlaceholder(variable_name="history")에 바인딩
        "input": "파이썬으로 웹 크롤링을 할 때 가장 많이 사용하는 라이브러리는 무엇인가요?" # HumanMessagePromptTemplate에 바인딩
    }
)

# 4. 결과 확인
print("--- 최종 PromptValue 내용 (Messages) ---")
print(final_prompt_value.messages)

print("\n--- 각 메시지 유형 확인 ---")
for message in final_prompt_value.messages:
    print(f"[{message.type.capitalize()}]: {message.content[:50]}...")